# TCA — Evaluate All Models
## AudioGuard FYP | TCA Track
**Purpose**: Run AFTER all T1–T6 notebooks have completed. Loads `training_summary.json`, compares all TCA models, plots a leaderboard bar chart, and prints the winner recommendation for the fusion layer.

**Prerequisite**: All of T1–T6 must have run and saved their results to `../../outputs/training_summary.json`

In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
print('✓ Imports loaded')

In [ ]:
# Cell 1 — Load training_summary.json and filter for TCA track
OUTPUTS_DIR = os.path.join('..', '..', 'outputs')
summary_path = os.path.join(OUTPUTS_DIR, 'training_summary.json')

if not os.path.exists(summary_path):
    raise FileNotFoundError(
        f'training_summary.json not found at {os.path.abspath(summary_path)}\n'
        'Run all T1–T6 notebooks first before running T_evaluate_all.'
    )

with open(summary_path, 'r') as f:
    all_results = json.load(f)

tca_results = [r for r in all_results if r.get('track') == 'TCA']
print(f'✓ Loaded {len(tca_results)} TCA model results')
print(f'Models found: {[r["model_id"] for r in tca_results]}')

if len(tca_results) < 6:
    print(f'\n⚠ Only {len(tca_results)}/6 TCA models found. Missing:')
    expected = {'T1', 'T2', 'T3', 'T4', 'T5', 'T6'}
    found = {r['model_id'] for r in tca_results}
    print(f'  Missing: {expected - found}')

df = pd.DataFrame(tca_results).sort_values('f1_macro', ascending=False)
print('\nTCA Results Summary:')
print(df[['model_id', 'model_name', 'accuracy', 'f1_macro', 'precision_macro', 'recall_macro', 'train_time_minutes', 'dataset']].to_string(index=False))

In [ ]:
# Cell 2 — Build and save TCA leaderboard CSV
leaderboard_path = os.path.join(OUTPUTS_DIR, 'tca_leaderboard.csv')
df_lb = df[['model_id', 'model_name', 'accuracy', 'f1_macro', 'precision_macro', 
             'recall_macro', 'train_time_minutes', 'peak_vram_gb', 'dataset', 'timestamp']]
df_lb = df_lb.reset_index(drop=True)
df_lb.index += 1  # rank starts at 1
df_lb.index.name = 'rank'
df_lb.to_csv(leaderboard_path)
print(f'✓ TCA leaderboard saved to {leaderboard_path}')
print(df_lb.to_string())

In [ ]:
# Cell 3 — Bar chart comparing F1 macro across all TCA models
model_labels = [f"{r['model_id']}\n{r['model_name'].split('/')[-1]}" for _, r in df.iterrows()]
f1_scores = df['f1_macro'].tolist()
colors = ['#2ecc71' if i == 0 else '#3498db' for i in range(len(f1_scores))]

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# F1 bar chart
bars = axes[0].barh(model_labels[::-1], f1_scores[::-1], color=colors[::-1])
axes[0].set_xlabel('F1 Macro Score')
axes[0].set_title('TCA Models — F1 Macro Comparison')
axes[0].axvline(x=max(f1_scores), color='red', linestyle='--', alpha=0.5, label='Best')
for bar, score in zip(bars, f1_scores[::-1]):
    axes[0].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{score:.4f}', va='center', fontsize=9)

# Accuracy bar chart
accs = df['accuracy'].tolist()
axes[1].barh(model_labels[::-1], accs[::-1], color='#9b59b6')
axes[1].set_xlabel('Accuracy')
axes[1].set_title('TCA Models — Accuracy Comparison')
for bar, score in zip(axes[1].patches, accs[::-1]):
    axes[1].text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
                 f'{score:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUTS_DIR, 'tca_leaderboard_chart.png'), dpi=150)
plt.show()
print(f'✓ Chart saved to {OUTPUTS_DIR}')

In [ ]:
# Cell 4 — Print winner
winner = df.iloc[0]
print('=' * 60)
print('BEST TCA MODEL')
print('=' * 60)
print(f'Model ID:       {winner["model_id"]}')
print(f'Model Name:     {winner["model_name"]}')
print(f'F1 Macro:       {winner["f1_macro"]:.4f}')
print(f'Accuracy:       {winner["accuracy"]*100:.2f}%'  if winner["accuracy"] <= 1 else f'Accuracy:   {winner["accuracy"]:.2f}%')
print(f'Precision:      {winner["precision_macro"]:.4f}')
print(f'Recall:         {winner["recall_macro"]:.4f}')
print(f'Train Time:     {winner["train_time_minutes"]:.1f} min')
print(f'Peak VRAM:      {winner.get("peak_vram_gb", 0):.2f} GB')
print()
print(f'RECOMMENDATION: Use {winner["model_id"]} ({winner["model_name"]}) in the fusion layer')
print('=' * 60)
print()
print('Full leaderboard (ranked by F1 macro):')
for rank, (_, row) in enumerate(df.iterrows(), 1):
    marker = ' ← WINNER' if rank == 1 else ''
    print(f'  {rank}. {row["model_id"]:3s} | {row["model_name"][:45]:45s} | F1: {row["f1_macro"]:.4f} | Acc: {row["accuracy"]:.4f}{marker}')